In [1]:
import os
import pandas as pd
from collections import defaultdict

In [2]:
# --- 1. PREPARACIÓN ---
# Ruta a la carpeta principal que contiene todos los datos.
ruta_principal = 'Contaminantes del aire Mexico'

In [3]:
# Usamos defaultdict para facilitar la agrupación.
# La llave será un conjunto congelado (frozenset) de nombres de columnas.
# El valor será una lista de rutas a los archivos que tienen esas columnas.
grupos_de_schemas = defaultdict(list)

In [4]:
print("Fase 1: Auditando la estructura de todos los archivos CSV...")

Fase 1: Auditando la estructura de todos los archivos CSV...


In [5]:
# --- FASE DE AUDITORÍA ---
# Recorremos todos los archivos para identificar sus columnas.
for dirpath, _, filenames in os.walk(ruta_principal):
    for filename in filenames:
        if filename.endswith('.csv'):
            ruta_completa = os.path.join(dirpath, filename)
            try:
                # Lectura ultra-rápida: solo leemos el encabezado para obtener las columnas.
                columnas = pd.read_csv(ruta_completa, nrows=0).columns
                
                # Creamos una "firma" única para este conjunto de columnas (schema).
                # Usamos frozenset porque es inmutable y puede ser una llave de diccionario.
                schema_firma = frozenset(columnas)
                
                # Agrupamos la ruta del archivo bajo su "firma" de schema.
                grupos_de_schemas[schema_firma].append(ruta_completa)
                
            except Exception as e:
                print(f"🟡 Advertencia: No se pudo leer el encabezado de '{filename}'. Error: {e}")

print("✅ Auditoría completada.")

✅ Auditoría completada.


In [6]:
# --- 2. ANALISISY REPORTE DE LOS RESULTADOS ---
print("\n--- Análisis de Estructuras de Archivos ---")
print(f"Se encontraron {len(grupos_de_schemas)} estructuras de columnas (schemas) diferentes.")

# Imprimimos un resumen de cada grupo encontrado.
for i, (schema, archivos) in enumerate(grupos_de_schemas.items()):
    print(f"\nSchema #{i+1}: Encontrado en {len(archivos)} archivos.")
    # Imprimimos las columnas de este schema para inspeccion.
    print(f"  Columnas: {sorted(list(schema))}")



--- Análisis de Estructuras de Archivos ---
Se encontraron 8 estructuras de columnas (schemas) diferentes.

Schema #1: Encontrado en 2 archivos.
  Columnas: [' pm10', ' pm25', 'date']

Schema #2: Encontrado en 4 archivos.
  Columnas: [' co', ' no2', ' o3', ' pm10', ' so2', 'date']

Schema #3: Encontrado en 2 archivos.
  Columnas: [' pm25', 'date']

Schema #4: Encontrado en 73 archivos.
  Columnas: [' co', ' no2', ' o3', ' pm10', ' pm25', ' so2', 'date']

Schema #5: Encontrado en 1 archivos.
  Columnas: [' o3', ' pm10', 'date']

Schema #6: Encontrado en 1 archivos.
  Columnas: [' no2', ' o3', ' pm10', 'date']

Schema #7: Encontrado en 1 archivos.
  Columnas: [' no2', ' o3', ' pm10', ' pm25', ' so2', 'date']

Schema #8: Encontrado en 2 archivos.
  Columnas: [' co', ' no2', ' o3', ' pm25', ' so2', 'date']


In [7]:
# --- 3. FASE DE CONSOLIDACIÓN ---
print("\n Fase 2: Consolidando archivos por cada grupo de schema...")

# Carpeta para guardar los nuevos archivos consolidados.
os.makedirs('consolidados', exist_ok=True)

# Iteramos sobre cada grupo de schema que encontramos.
for i, (schema, archivos) in enumerate(grupos_de_schemas.items()):
    
    lista_de_dataframes = []
    print(f"\n--- Procesando Schema #{i+1} ({len(archivos)} archivos) ---")
    
    for ruta_archivo in archivos:
        try:
            # Ahora sí, leemos el archivo CSV completo.
            df = pd.read_csv(ruta_archivo)
            
            # --- Enriquecimiento de datos (igual que antes) ---
            df['estado'] = os.path.basename(os.path.dirname(ruta_archivo))
            df['fuente_archivo'] = os.path.basename(ruta_archivo)
            
            lista_de_dataframes.append(df)
        except Exception as e:
            print(f"🟡 Advertencia: No se pudo procesar el archivo '{os.path.basename(ruta_archivo)}'. Error: {e}")

    # Concatenamos todos los DataFrames de la lista en uno solo.
    if lista_de_dataframes:
        df_consolidado = pd.concat(lista_de_dataframes, ignore_index=True)
        
        # Guardamos el resultado en un nuevo archivo CSV.
        nombre_salida = f'consolidados/consolidado_schema_{i+1}.csv'
        df_consolidado.to_csv(nombre_salida, index=False)
        
        print(f"✅ Grupo consolidado y guardado como '{nombre_salida}' con {len(df_consolidado)} filas.")
    else:
        print("❌ No se pudo consolidar este grupo debido a errores de lectura.")

print("\n ¡Proceso de consolidación finalizado!")


 Fase 2: Consolidando archivos por cada grupo de schema...

--- Procesando Schema #1 (2 archivos) ---
✅ Grupo consolidado y guardado como 'consolidados/consolidado_schema_1.csv' con 4012 filas.

--- Procesando Schema #2 (4 archivos) ---
✅ Grupo consolidado y guardado como 'consolidados/consolidado_schema_2.csv' con 10986 filas.

--- Procesando Schema #3 (2 archivos) ---
✅ Grupo consolidado y guardado como 'consolidados/consolidado_schema_3.csv' con 3806 filas.

--- Procesando Schema #4 (73 archivos) ---
✅ Grupo consolidado y guardado como 'consolidados/consolidado_schema_4.csv' con 188626 filas.

--- Procesando Schema #5 (1 archivos) ---
✅ Grupo consolidado y guardado como 'consolidados/consolidado_schema_5.csv' con 1649 filas.

--- Procesando Schema #6 (1 archivos) ---
✅ Grupo consolidado y guardado como 'consolidados/consolidado_schema_6.csv' con 3104 filas.

--- Procesando Schema #7 (1 archivos) ---
✅ Grupo consolidado y guardado como 'consolidados/consolidado_schema_7.csv' con 256